# SCARF

## Model

In [ ]:
# ============================================================
# BUILT-IN SCARF IMPLEMENTATION
# ============================================================

class SCARF(nn.Module):
    """Self-Supervised Contrastive Learning for Tabular Data"""

    def __init__(self, input_dim, emb_dim=128, encoder_depth=3, head_depth=2,
                 dropout_rate=0.15, corruption_rate=0.5):
        super().__init__()

        self.input_dim = input_dim
        self.corruption_rate = corruption_rate

        # Encoder (MLP)
        encoder_layers = []
        prev_dim = input_dim
        for _ in range(encoder_depth):
            encoder_layers.append(nn.Linear(prev_dim, emb_dim))
            encoder_layers.append(nn.LayerNorm(emb_dim))
            encoder_layers.append(nn.GELU())
            encoder_layers.append(nn.Dropout(dropout_rate))
            prev_dim = emb_dim

        self.encoder = nn.Sequential(*encoder_layers)

       # Projection head
        head_layers = []
        prev_dim = emb_dim
        for _ in range(head_depth):
            head_layers.append(nn.Linear(prev_dim, emb_dim))
            head_layers.append(nn.LayerNorm(emb_dim))
            head_layers.append(nn.GELU())
            head_layers.append(nn.Dropout(dropout_rate))
            prev_dim = emb_dim

        head_layers.append(nn.Linear(emb_dim, emb_dim))
        self.projection_head = nn.Sequential(*head_layers)

    def corrupt(self, x):
        """Random feature replacement (corruption)"""
        x_corrupted = x.clone()
        mask = torch.rand(x.shape, device=x.device) < self.corruption_rate

        # For each feature column, shuffle values among the batch
        for col in range(self.input_dim):
            random_indices = torch.randperm(len(x), device=x.device)
            random_values = x[random_indices, col]
            x_corrupted[mask[:, col], col] = random_values[mask[:, col]]

        return x_corrupted

    def forward(self, x):
        """Forward pass: returns embeddings for anchor and corrupted views"""
        x_corrupted = self.corrupt(x)

        # Compute embeddings through encoder + projection head
        emb_anchor = self.projection_head(self.encoder(x))
        emb_positive = self.projection_head(self.encoder(x_corrupted))

        return emb_anchor, emb_positive

    def get_embeddings(self, x):
        """Extract embeddings from the encoder (without projection head)"""
        with torch.no_grad():
            return self.encoder(x)


def get_scarf_embeddings(model, data_loader, device):
    """Extract embeddings from a trained SCARF model for all samples in a DataLoader"""
    model.eval()
    embeddings = []

    with torch.no_grad():
        for batch in data_loader:
            # Handle both (features, labels) tuples and feature-only batches
            if isinstance(batch, (list, tuple)):
                x = batch[0]
            else:
                x = batch

            x = x.float().to(device)
            emb = model.get_embeddings(x)
            embeddings.append(emb.cpu())

    # Return empty array if no embeddings were collected
    if len(embeddings) == 0:
        return np.array([])
    return torch.cat(embeddings, dim=0).numpy()


print("SCARF implementation ready")

## Pretraining

In [ ]:
# ============================================================
# SCARF: TRAINING (with early stopping)
# ============================================================

print("\n" + "="*60)
print("SCARF: CONTRASTIVE PRETRAINING (built-in)")
print("="*60)

# Configuration parameters
SCARF_CONFIG = {
    'emb_dim': 256,
    'encoder_depth': 4,
    'head_depth': 2,
    'dropout_rate': 0.1,
    'corruption_rate': 0.6,
    'temperature': 1,
    'batch_size': 128,
    'epochs': 200,
    'learning_rate': 1e-3,
    'weight_decay': 1e-6,
    'eta_min': 1e-4,
    'clip_grad': 1.0,
    'patience': 200,  # Without early stopping
}

print(f"Configuration:")
for key, val in SCARF_CONFIG.items():
    print(f"  {key}: {val}")

# Prepare pretrain data
X_pretrain_scarf = X_pretrain.values if hasattr(X_pretrain, 'values') else X_pretrain
print(f"\nPretrain size: {len(X_pretrain_scarf)}")

# DataLoader
dataset = TensorDataset(torch.FloatTensor(X_pretrain_scarf))
dataloader = DataLoader(dataset, batch_size=SCARF_CONFIG['batch_size'], shuffle=True)

# Model initialization
device_scarf = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_scarf = SCARF(
    input_dim=X_pretrain_scarf.shape[1],
    emb_dim=SCARF_CONFIG['emb_dim'],
    encoder_depth=SCARF_CONFIG['encoder_depth'],
    head_depth=SCARF_CONFIG['head_depth'],
    dropout_rate=SCARF_CONFIG['dropout_rate'],
    corruption_rate=SCARF_CONFIG['corruption_rate']
).to(device_scarf)

print(f"Model parameters: {sum(p.numel() for p in model_scarf.parameters()):,}")
print(f"Device: {device_scarf}")

# Loss function (NT-Xent as used in CLT_TAB)
criterion_scarf = NTXent(temperature=SCARF_CONFIG['temperature'])

# Optimizer (AdamW as used in CLT_TAB)
optimizer = torch.optim.AdamW(
    model_scarf.parameters(),
    lr=SCARF_CONFIG['learning_rate'],
    weight_decay=SCARF_CONFIG['weight_decay']
)

# Scheduler (CosineAnnealing as used in CLT_TAB)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=SCARF_CONFIG['epochs'],
    eta_min=SCARF_CONFIG['eta_min']
)

# Training
print("\n" + "-"*40)
print("Training SCARF...")
print("-"*40)

from tqdm.auto import tqdm

loss_history_scarf = []
best_loss = float('inf')
patience_counter = 0
best_model_state = None

model_scarf.train()
for epoch in tqdm(range(1, SCARF_CONFIG['epochs'] + 1), desc="Training SCARF"):
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        x = batch[0].to(device_scarf)

        # Forward pass
        optimizer.zero_grad()
        z_i, z_j = model_scarf(x)
        loss = criterion_scarf(z_i, z_j)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model_scarf.parameters(), SCARF_CONFIG['clip_grad'])

        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    # Update learning rate scheduler
    scheduler.step()

    # Compute average loss for the epoch
    avg_loss = total_loss / num_batches
    loss_history_scarf.append(avg_loss)

    # Early stopping logic
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        # Save best model state (move to CPU to save GPU memory)
        best_model_state = {k: v.cpu().clone() for k, v in model_scarf.state_dict().items()}
    else:
        patience_counter += 1

    # Progress logging every 10 epochs
    if epoch % 10 == 0:
        current_lr = scheduler.get_last_lr()[0]
        tqdm.write(f"Epoch {epoch:3d}/{SCARF_CONFIG['epochs']} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f} | Patience: {patience_counter}/{SCARF_CONFIG['patience']}")

    # Stop training if patience is exceeded
    if patience_counter >= SCARF_CONFIG['patience']:
        tqdm.write(f"\n⚠️ Early stopping at epoch {epoch} (loss didn't improve for {SCARF_CONFIG['patience']} epochs)")
        break

# Load the best model
if best_model_state is not None:
    model_scarf.load_state_dict(best_model_state)
    model_scarf = model_scarf.to(device_scarf)
    print(f"\n Loaded best model with loss: {best_loss:.4f}")

print("\n SCARF pretraining completed!")

# Plot training loss curve
plt.figure(figsize=(10, 5))
plt.plot(loss_history_scarf, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('SCARF: Contrastive Learning Loss (with Early Stopping)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Downstream (Logistic Regression)

In [ ]:
# ============================================================
# SCARF: GET EMBEDDINGS
# ============================================================

print("\n" + "="*60)
print("SCARF: GENERATING EMBEDDINGS")
print("="*60)

# Create datasets for finetuning and test sets
ft_dataset = TensorDataset(torch.FloatTensor(ft_pool_data.values if hasattr(ft_pool_data, 'values') else ft_pool_data))
test_dataset = TensorDataset(torch.FloatTensor(test_data.values if hasattr(test_data, 'values') else test_data))

# Create data loaders (no shuffling for inference)
ft_loader = DataLoader(ft_dataset, batch_size=SCARF_CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=SCARF_CONFIG['batch_size'], shuffle=False)

def get_scarf_embeddings(model, data_loader, device):
    """Extract embeddings from a trained SCARF model for all samples in a DataLoader"""
    model.eval()
    embeddings = []

    with torch.no_grad():
        for batch in data_loader:
            # Handle both (features, labels) tuples and feature-only batches
            if isinstance(batch, (list, tuple)):
                x = batch[0]
            else:
                x = batch

            x = x.float().to(device)
            emb = model.get_embeddings(x)
            embeddings.append(emb.cpu())

    if len(embeddings) == 0:
        return np.array([])
    return torch.cat(embeddings, dim=0).numpy()

# Generate embeddings for both sets
ft_embeddings_scarf = get_scarf_embeddings(model_scarf, ft_loader, device_scarf)
test_embeddings_scarf = get_scarf_embeddings(model_scarf, test_loader, device_scarf)

print(f"Fine-tune embeddings shape: {ft_embeddings_scarf.shape}")
print(f"Test embeddings shape: {test_embeddings_scarf.shape}")

# ============================================================
# SCARF: DOWNSTREAM EVALUATION (Logistic Regression)
# ============================================================

print("\n" + "="*60)
print("SCARF: DOWNSTREAM EVALUATION (Logistic Regression)")
print("="*60)

# Logistic regression hyperparameter grid
param_grid_lr = {
    'penalty': ['l2'],
    'C': np.logspace(-2, 0, 7).tolist(),
    'solver': ['lbfgs']
}

# Evaluation metrics
scoring = {
    'f1': make_scorer(f1_score),
    'auprc': make_scorer(average_precision_score),
    'recall': make_scorer(recall_score)
}

# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

# Grid search with cross-validation
grid_search_scarf = GridSearchCV(
    estimator=LogisticRegression(max_iter=10000, class_weight='balanced', random_state=seed),
    param_grid=param_grid_lr,
    scoring=scoring,
    refit='auprc',
    cv=skf,
    verbose=1,
    n_jobs=-1
)

# Train the classifier on finetune embeddings
grid_search_scarf.fit(ft_embeddings_scarf, finetune_target)

# Extract best results
best_idx_scarf = grid_search_scarf.best_index_
print(f"\nBest parameters: {grid_search_scarf.best_params_}")
print(f"Best AUPRC (CV): {grid_search_scarf.best_score_:.4f}")
print(f"F1 (CV): {grid_search_scarf.cv_results_['mean_test_f1'][best_idx_scarf]:.4f}")
print(f"Recall (CV): {grid_search_scarf.cv_results_['mean_test_recall'][best_idx_scarf]:.4f}")

# Evaluate on test set
y_pred_scarf = grid_search_scarf.predict(test_embeddings_scarf)
probs_scarf = grid_search_scarf.predict_proba(test_embeddings_scarf)[:, 1]

print("\n--- Test Set ---")
print(f"AUPRC: {average_precision_score(test_target, probs_scarf):.4f}")
print(f"F1: {f1_score(test_target, y_pred_scarf):.4f}")
print(f"Recall: {recall_score(test_target, y_pred_scarf):.4f}")
print(classification_report(test_target, y_pred_scarf))

# ============================================================
# SCARF: SUMMARY
# ============================================================
print("\n" + "="*60)
print("SCARF: SUMMARY")
print("="*60)
print(f"{'Metric':<20} {'Value':<12}")
print("-"*32)
print(f"{'Test AUPRC':<20} {average_precision_score(test_target, probs_scarf):<12.4f}")
print(f"{'Test F1':<20} {f1_score(test_target, y_pred_scarf):<12.4f}")
print(f"{'Test Recall':<20} {recall_score(test_target, y_pred_scarf):<12.4f}")